<a href="https://colab.research.google.com/github/sayedHoosini/AI_Portfolio_ITAI2372/blob/main/L05_Notebook__Hoosini_ITAI2377.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Title and Introduction
##Data Preprocessing Lab: Generative AI
###    Welcome to the Data Preprocessing Lab for Generative AI!
   In this lab, you'll get hands-on experience with key preprocessing techniques for both text and (optionally) image data.

   **Learning Objectives:**


*  Understand and apply core data preprocessing techniques.
*  Explore word embedding techniques (Word2Vec/GloVe, BERT).
*  Analyze the impact of preprocessing choices on data quality and
   model suitability. List item
*   Practice using cosine similarity for comparing embeddings.

## Part 1: Environment Setup

First, we'll install and import all necessary libraries. Run the following cell to set up your environment.

In [103]:
# SECTION 1: Environment Setup
#############################
# This cell installs and imports all necessary libraries for our text preprocessing pipeline.
# We'll be using:
# - pandas & numpy: for data manipulation
# - nltk: for natural language processing tasks
# - scikit-learn: for machine learning utilities
# - transformers & torch: for BERT embeddings
# - gensim: for word embeddings (Word2Vec/GloVe)

# Install required packages
%pip install pandas numpy nltk scikit-learn transformers torch datasets gensim

# TODO: Import the required libraries
# Hint: You need pandas, numpy, nltk, and sklearn components
# YOUR CODE HERE - import the basic libraries
import pandas as pd
import numpy as np
import nltk
# sklearn utilities
# sklearn imports for machine learning utilities
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
# Add more imports as needed...
# Advanced imports
# These are more advanced imports you'll need later
from transformers import BertTokenizer, BertModel
import torch
import gensim.downloader as api
from gensim.models import KeyedVectors

# Download required NLTK data
# These are necessary for tokenization, stop words, and lemmatization
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab') # Added as per error message

print("Setup complete! All required libraries have been imported.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Setup complete! All required libraries have been imported.


[nltk_data]   Package punkt_tab is already up-to-date!


## Part 2: Loading and Exploring the BBC News Dataset

We'll now load the BBC News dataset  you used in previous assignments, and perform initial exploration of its contents.

In [104]:
# SECTION 2: Data Loading and Initial Exploration
##############################################
# Here we load the BBC News dataset and perform initial analysis
# Understanding our data is crucial before applying any preprocessing

# Load the dataset
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

print("Loading BBC News dataset...")

try:
    # Load the BBC News dataset from Hugging Face
    # The 'yikuan8/bbc-news-classify' dataset is a good candidate for this.
    dataset = load_dataset("SetFit/bbc-news")
    # Convert to pandas DataFrame, focusing on the 'train' split if available
    if 'train' in dataset:
        df = dataset["train"].to_pandas()
    else:
        # Fallback if only one split exists, or to handle other structures
        df = next(iter(dataset.values())).to_pandas()

    print("Dataset loaded successfully into DataFrame using 'datasets' library.")

except Exception as e:
    print(f"Error loading dataset using 'datasets' library: {e}")
    print("Please ensure the dataset name is correct and accessible.")
    raise

# The dataset from 'yikuan8/bbc-news-classify' has 'text' and 'labels' columns.
# We need to rename 'labels' to 'category' for consistency with the rest of the notebook.
df = df.rename(columns={'label': 'category'})


# TODO: Perform basic data exploration
# TASK 1: Display the first few rows and basic information about the dataset
# Hint: Use pandas' head(), info(), and describe() methods
# YOUR CODE HERE
print("\nFirst 5 rows of the dataset:")
print(df.head())
print("\nDataset Info:")
df.info()
print("\nDataset Description (including all columns):")
print(df.describe(include='all'))


Loading BBC News dataset...
Dataset loaded successfully into DataFrame using 'datasets' library.

First 5 rows of the dataset:
                                                text  category     label_text
0  wales want rugby league training wales could f...         2          sport
1  china aviation seeks rescue deal scandal-hit j...         1       business
2  rock band u2 break ticket record u2 have smash...         3  entertainment
3  markets signal brazilian recovery the brazilia...         1       business
4  tough rules for ringtone sellers firms that fl...         0           tech

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1225 entries, 0 to 1224
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        1225 non-null   object
 1   category    1225 non-null   int64 
 2   label_text  1225 non-null   object
dtypes: int64(1), object(2)
memory usage: 28.8+ KB

Dataset Description (including a

# Comprehension Questions - Data Exploration

Answer the following questions based on the dataset exploration above:

1. What are the dimensions of our dataset?
2. How many different categories are there in the news articles?
3. Is the dataset balanced across categories? Why might this matter?
4. Are there any missing values that need to be addressed?

## Part 3: Text Preprocessing

We'll now implement basic text preprocessing steps to clean our data.

In [105]:
# SECTION 3: Text Cleaning and Preprocessing
########################################
# This section implements fundamental text preprocessing steps:
# 1. Converting to lowercase (why? -> maintains consistency)
# 2. Removing special characters (why? -> reduces noise)
# 3. Handling whitespace (why? -> standardizes format)

import re

def clean_text(text):
    """
    Performs basic text cleaning operations.

    Parameters:
    text (str): Input text to be cleaned

    Returns:
    str: Cleaned text
    """
    # TODO: Implement the following steps:
    # 1. Convert to lowercase
    # 2. Remove URLs and emails
    # 3. Remove special characters but keep sentence structure
    # 4. Remove extra whitespace
    # Hint: Use string methods and regular expressions

    # YOUR CODE HERE
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove emails
    text = re.sub(r'\S*@\S*\s?', '', text)
    # Remove special characters, keep alphanumeric and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Keeping only letters and spaces as tokenization later uses isalpha()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test the function with a sample
sample_text = "Hello, World! This is a TEST... 123 http://example.com email@example.com"
print("Original:", sample_text)
print("Cleaned:", clean_text(sample_text))

# Apply to the entire dataset
df['cleaned_text'] = df['text'].apply(clean_text)


Original: Hello, World! This is a TEST... 123 http://example.com email@example.com
Cleaned: hello world this is a test


## Part 4: Tokenization and Advanced Processing

Now we'll tokenize our text and apply more advanced preprocessing techniques including:
- Tokenization
- Stop word removal
- Lemmatization

In [106]:
# SECTION 4: Tokenization and Advanced Processing
#############################################
# This section implements more sophisticated NLP techniques:
# - Tokenization: splitting text into words
# - Stop word removal: removing common words
# - Lemmatization: reducing words to their base form
# Check if you do not need to install any additional libraries
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Initialize our tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def tokenize_and_process(text):
    # TODO: Implement the following steps:
    # 1. Tokenize the text
    # 2. Remove stop words
    # 3. Apply lemmatization
    # Hint: Use the initialized stop_words and lemmatizer

    # YOUR CODE HERE
    tokens = word_tokenize(text.lower())
    # Replace with actual tokenization
    tokens = [word for word in tokens if word not in stop_words and word.isalpha()]
    # Step 3: Lemmatize the remaining words
    processed_tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Replace with processed tokens

    return processed_tokens

# Test the function
sample_text = "The quick brown foxes are jumping over the lazy dogs"
processed_result = tokenize_and_process(sample_text)
print("Original:", sample_text)
print("Processed:", processed_result)

Original: The quick brown foxes are jumping over the lazy dogs
Processed: ['quick', 'brown', 'fox', 'jumping', 'lazy', 'dog']


## Part 5: Word Embeddings with GloVe

We'll now generate word embeddings using pre-trained GloVe vectors. These embeddings will help us capture semantic relationships between words in our articles.

In [107]:
# SECTION 5: Word Embeddings with GloVe
###################################
# This section generates word embeddings using pre-trained GloVe vectors
# Word embeddings capture semantic relationships between words
# by representing them as dense vectors in a high-dimensional space

# Load pre-trained GloVe embeddings
glove_model = api.load("glove-wiki-gigaword-100")

# SECTION 5: Word Embeddings with GloVe
###################################

# Load pre-trained GloVe embeddings
print("GloVe model loaded successfully")

def get_word2vec_embedding(text, model):
    """
    Generates document embeddings by averaging word vectors.

    Parameters:
    text (str): Input text
    model: Pre-trained word embedding model

    Returns:
    numpy.array: Document embedding vector
    """
    # TODO: Implement the following steps:
    # 1. Tokenize the input text
    # 2. Get embedding for each token
    # 3. Average the embeddings
    # Hint: Handle words not in vocabulary

    # YOUR CODE HERE
    tokens = word_tokenize(str(text).lower())

    # Step 2: Get embeddings for each token that exists in the model
    embeddings = [model[word] for word in tokens if word in model]
    # The following line was redundant and overwriting the embeddings list. Removed.
    embeddings = [model[word] for word in tokens if word in model]

    # Step 3: average the embeddings
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(model.vector_size)
# Apply to a sample of the dataset
sample_size = 100
sample_df = df.head(sample_size).copy()
sample_df['glove_embedding'] = sample_df['cleaned_text'].apply(
    lambda x: get_word2vec_embedding(x, glove_model)
)


GloVe model loaded successfully


## Part 6: BERT Embeddings

Now we'll use BERT to generate contextual embeddings. BERT provides context-aware embeddings that can capture more nuanced relationships in the text.

In [108]:
# SECTION 6: BERT Embeddings
#########################
# This section implements BERT (Bidirectional Encoder Representations from Transformers)
# BERT provides context-aware embeddings, meaning the same word can have different
# embeddings based on its context in the sentence.
# Key differences from GloVe:
# - Contextual (words have different vectors based on context)
# - Deep bidirectional (considers both left and right context)
# - Pre-trained on massive datasets


# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_embedding(text, max_length=512):
    """
    Generates BERT embeddings for a text.

    Parameters:
    text (str): Input text
    max_length (int): Maximum sequence length for BERT

    Returns:
    numpy.array: BERT embedding vector
    """
    # TODO: Implement the following steps:
    # 1. Tokenize the text using BERT tokenizer
    # 2. Generate BERT embeddings
    # 3. Extract the [CLS] token embedding
    # Hint: Use tokenizer() and model() functions

    # YOUR CODE HERE
    # Step 1: Tokenize
    inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=max_length
    )

    # Step 2: Generate embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Step 3: Extract [CLS] token embedding
    sentence_embedding = outputs.last_hidden_state[:, 0, :].numpy()

    return sentence_embedding

# Test the function
test_text = "This is a test sentence for BERT embeddings."
bert_embedding = get_bert_embedding(test_text)
print("BERT embedding shape:", bert_embedding.shape)

# Apply BERT embeddings to a sample of the dataset
sample_df['bert_embedding'] = sample_df['cleaned_text'].apply(get_bert_embedding)
print("BERT embeddings applied to sample_df.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT embedding shape: (1, 768)
BERT embeddings applied to sample_df.


## Part 7: Comparing Embeddings

Let's analyze how well our different embedding methods capture semantic relationships by comparing similarities between articles in the same and different categories.

In [109]:
# SECTION 7: Similarity Analysis
############################
# This section implements methods to compare different embedding approaches
# We'll analyze how well each embedding type captures semantic relationships
# by comparing similarities between articles in the same and different categories
# SECTION 6: BERT Embeddings
#########################

# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_embedding(text, max_length=512):
    """
    Generates BERT embeddings for a text.

    Parameters:
    text (str): Input text
    max_length (int): Maximum sequence length for BERT

    Returns:
    numpy.array: BERT embedding vector
    """
    # TODO: Implement the following steps:
    # 1. Tokenize the text using BERT tokenizer
    # 2. Generate BERT embeddings
    # 3. Extract the [CLS] token embedding
    # Hint: Use tokenizer() and model() functions

    # YOUR CODE HERE
    # Step 1: Tokenize
    inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=max_length
    )

    # Step 2: Generate embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Step 3: Extract [CLS] token embedding
    sentence_embedding = outputs.last_hidden_state[:, 0, :].numpy()

    return sentence_embedding

# Test the function
test_text = "This is a test sentence for BERT embeddings."
bert_embedding = get_bert_embedding(test_text)
print("BERT embedding shape:", bert_embedding.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT embedding shape: (1, 768)


In [110]:
# Generate BERT embeddings for the dataset
sample_df["bert_embedding"] = sample_df["cleaned_text"].apply(
    lambda x: get_bert_embedding(x)
)

print("BERT embeddings created.")
print(sample_df["bert_embedding"].iloc[0].shape)

BERT embeddings created.
(1, 768)


 ##SECTION 8: Detailed Similarity Analysis

In [111]:
# SECTION 8: Detailed Similarity Analysis
####################################
# This section analyzes how well our embeddings capture
# semantic relationships between articles

# SECTION 8: Detailed Similarity Analysis
####################################

def analyze_category_similarities(similarities, categories):
    """
    Analyzes similarities within and across categories.

    Parameters:
    similarities (numpy.array): Similarity matrix
    categories (list): List of category labels

    Returns:
    dict: Statistics about similarities
    """
    # TODO: Implement the following analysis:
    # 1. Separate similarities into same-category and different-category groups
    # 2. Calculate statistics for each group
    # Hint: Use nested loops to compare categories

    # YOUR CODE HERE
    same_category_sims = []
    diff_category_sims = []

    n_samples = similarities.shape[0]
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            if categories[i] == categories[j]:
                same_category_sims.append(similarities[i, j])
            else:
                diff_category_sims.append(similarities[i, j])

    # Your implementation here

    return {
        'same_category': {
            'mean': np.mean(same_category_sims) if same_category_sims else 0,
            'std': np.std(same_category_sims) if same_category_sims else 0
        },
        'diff_category': {
            'mean': np.mean(diff_category_sims) if diff_category_sims else 0,
            'std': np.std(diff_category_sims) if diff_category_sims else 0
        }
    }

# Before calling analyze_category_similarities, we need to compute the similarity matrices.
# This assumes sample_df, sample_df['glove_embedding'], and sample_df['bert_embedding'] are available.

# Ensure embeddings are in the correct 2D format for cosine_similarity
# Filter out potential empty embeddings to avoid issues with np.vstack
glove_valid_embeddings = [emb for emb in sample_df['glove_embedding'].values if emb is not None and emb.size > 0]
bert_valid_embeddings = [arr.flatten() for arr in sample_df['bert_embedding'].values if arr is not None and arr.size > 0]

if not glove_valid_embeddings:
    print("Warning: No valid GloVe embeddings found to calculate similarities.")
    glove_similarities = np.array([[0.]]) # Default to a minimal similarity matrix
else:
    glove_embeddings_matrix = np.vstack(glove_valid_embeddings)
    print(f"GloVe embeddings matrix shape: {glove_embeddings_matrix.shape}")
    glove_similarities = cosine_similarity(glove_embeddings_matrix)

if not bert_valid_embeddings:
    print("Warning: No valid BERT embeddings found to calculate similarities.")
    bert_similarities = np.array([[0.]]) # Default to a minimal similarity matrix
else:
    bert_embeddings_matrix = np.vstack(bert_valid_embeddings)
    print(f"BERT embeddings matrix shape: {bert_embeddings_matrix.shape}")
    bert_similarities = cosine_similarity(bert_embeddings_matrix)

# Analyze both embedding types
glove_analysis = analyze_category_similarities(glove_similarities, sample_df['category'].values)
bert_analysis = analyze_category_similarities(bert_similarities, sample_df['category'].values)

# Print results
print("=== Similarity Analysis Results ===")
# TODO: Format and display the analysis results
# YOUR CODE HERE
print("\nGloVe Embeddings Analysis:")
print(f"  Mean similarity within same category: {glove_analysis['same_category']['mean']:.4f}")
print(f"  Std deviation within same category: {glove_analysis['same_category']['std']:.4f}")
print(f"  Mean similarity across different categories: {glove_analysis['diff_category']['mean']:.4f}")
print(f"  Std deviation across different categories: {glove_analysis['diff_category']['std']:.4f}")

print("\nBERT Embeddings Analysis:")
print(f"  Mean similarity within same category: {bert_analysis['same_category']['mean']:.4f}")
print(f"  Std deviation within same category: {bert_analysis['same_category']['std']:.4f}")
print(f"  Mean similarity across different categories: {bert_analysis['diff_category']['mean']:.4f}")
print(f"  Std deviation across different categories: {bert_analysis['diff_category']['std']:.4f}")


GloVe embeddings matrix shape: (100, 100)
BERT embeddings matrix shape: (100, 768)
=== Similarity Analysis Results ===

GloVe Embeddings Analysis:
  Mean similarity within same category: 0.9715
  Std deviation within same category: 0.0153
  Mean similarity across different categories: 0.9515
  Std deviation across different categories: 0.0220

BERT Embeddings Analysis:
  Mean similarity within same category: 0.8157
  Std deviation within same category: 0.0477
  Mean similarity across different categories: 0.7646
  Std deviation across different categories: 0.0430


## Part 9 Vizualizations

In [112]:
# SECTION 9: Visualization of Results
################################
# This section creates visualizations to help us understand
# the differences between our embedding approaches
################################

def plot_similarity_distributions(glove_sims, bert_sims, categories):
    """
    Creates visualization comparing GloVe and BERT similarity distributions.

    Parameters:
    glove_sims (numpy.array): GloVe similarity matrix
    bert_sims (numpy.array): BERT similarity matrix
    categories (list): Category labels
    """
    # TODO: Create the following visualizations:
    # 1. Histogram or density plot of similarities
    # 2. Box plot comparing same-category vs different-category similarities
    # 3. Add appropriate labels and titles
    # Hint: Use plt.subplots() for multiple plots

    # YOUR CODE HERE
    plt.figure(figsize=(15, 5))

    # Add your visualization code here

    plt.tight_layout()
    plt.show()

# Create visualizations
plot_similarity_distributions(glove_similarities,
                            bert_similarities,
                            sample_df['category'])

<Figure size 1500x500 with 0 Axes>

## Part 10 Statistical Comparison

In [113]:
# SECTION 10: Statistical Comparison
###############################
# This section performs statistical tests to compare
# the effectiveness of different embedding approaches
###############################

def compare_embedding_methods(glove_analysis, bert_analysis):
    """
    Performs statistical comparison of embedding methods.

    Parameters:
    glove_analysis (dict): GloVe similarity analysis results
    bert_analysis (dict): BERT similarity analysis results
    """
    # TODO: Implement the following analyses:
    # 1. Calculate effect sizes for both methods
    # 2. Perform statistical tests comparing the methods
    # 3. Summarize the findings
    # Hint: Consider using t-tests or Mann-Whitney U tests

    # YOUR CODE HERE

    # Print summary of findings
    print("=== Statistical Comparison Results ===")
    # Add your summary here

# Run comparison
compare_embedding_methods(glove_analysis, bert_analysis)


=== Statistical Comparison Results ===


## Part 11 Metrics and Evaluation

In [114]:
# SECTION 11: Performance Evaluation
##############################
# This section calculates various metrics to evaluate
# the quality of our embeddings

##############################

def calculate_metrics(similarities, categories):
    """
    Calculates performance metrics for embeddings.

    Parameters:
    similarities (numpy.array): Similarity matrix
    categories (list): Category labels

    Returns:
    dict: Dictionary of performance metrics
    """
    # TODO: Implement various metrics such as:
    # 1. Category separation score
    # 2. Silhouette score
    # 3. Custom metrics you design
    # Hint: Consider what makes embeddings "good" for your use case

    # YOUR CODE HERE
    metrics = {}

    return metrics

# Calculate metrics for both embedding types
glove_metrics = calculate_metrics(glove_similarities, sample_df['category'])
bert_metrics = calculate_metrics(bert_similarities, sample_df['category'])

# Display results
print("=== Performance Metrics ===")
# TODO: Format and display the metrics
# YOUR CODE HERE


=== Performance Metrics ===


## Part 12: Final Analysis Questions

1. Compare the similarity distributions for GloVe and BERT embeddings:
   - Which method better distinguishes between same-category and different-category articles?
   - What might explain the differences in performance?

2. Based on the visualizations:
   - What patterns do you notice in the similarity distributions?
   - Are there any unexpected results?

3. Considering the entire preprocessing pipeline:
   - Which steps had the biggest impact on the final results?
   - What additional preprocessing steps might improve the results?
   - How would you modify this pipeline for different types of text data?

4. Ethical Considerations:
   - What biases might be present in our preprocessing pipeline?
   - How might these biases affect the analysis of news articles?
   - What steps could we take to mitigate these biases?

###Assessment Criteria:

  * Correct implementation of cosine similarity
  *Proper normalization of embeddings
  *Effective visualization of results


##Grading Rubric

* Environment Setup: 10%
* Data Exploration: 15%
* Text Preprocessing: 20%
* Word Embeddings Implementation: 25%
* Similarity Analysis: 20%
Final Analysis & Discussion: 10%

##Common Issues and Solutions

1. Memory Issues:

* Implement batch processing for large datasets
* Use appropriate data types (float32 vs float64)
* Clear unused variables and call garbage collection


2. Performance Optimization:

* Vectorize operations where possible
* Use appropriate batch sizes for BERT
* Implement caching for embeddings


3. Error Handling:

* Implement robust error checking
* Provide clear error messages
* Handle edge cases appropriately